## Question 6: Implement a basic Langchain pipeline using OpenAI’s LLM to answer questions based on a user input prompt.


### I tried OpenAI's API free trial key  but it keep throughing error.
### That is why I am using Groq API key.

In [1]:
!pip install -q langchain langchain-groq -q

In [2]:
from google.colab import userdata

my_api_key = userdata.get('Groq_api')

In [3]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model='llama-3.3-70b-versatile',
    temperature=0.2,
    api_key=my_api_key
)

In [4]:
question = input("Ask a question: ")
answer = llm.invoke(question)
print("\nAnswer:", answer.content)

Ask a question: Who is president of UK?

Answer: The United Kingdom (UK) is a parliamentary democracy and a constitutional monarchy, which means it has a monarch (currently King Charles III) as its head of state, but not a president. The head of government is the Prime Minister, who is currently Rishi Sunak. 

So, to summarize: 
- Head of State: King Charles III (monarch)
- Head of Government: Rishi Sunak (Prime Minister)


## Question 7: Integrate Langchain with a third-party API (e.g., weather, news) and show how responses can be generated via LLMs.


In [5]:
# Integrating an API key for weather forecasting

import requests
from langchain_core.tools import tool

@tool
def get_weather(city: str) -> str:
    """Get the current weather for a given city name."""
    # Step A: convert city name to lat/long (free, no key needed)
    geo_url = "https://geocoding-api.open-meteo.com/v1/search"
    geo_resp = requests.get(geo_url, params={"name": city, "count": 1}).json()

    if "results" not in geo_resp or len(geo_resp["results"]) == 0:
        return f"Could not find location data for '{city}'."

    location = geo_resp["results"][0]
    lat, lon = location["latitude"], location["longitude"]
    resolved_name = location["name"]
    country = location.get("country", "")

    # Step B: fetch current weather for those coordinates
    weather_url = "https://api.open-meteo.com/v1/forecast"
    weather_resp = requests.get(weather_url, params={
        "latitude": lat,
        "longitude": lon,
        "current_weather": True
    }).json()

    current = weather_resp.get("current_weather", {})
    if not current:
        return f"Weather data unavailable for {resolved_name}."

    temp = current.get("temperature")
    windspeed = current.get("windspeed")

    return (f"Current weather in {resolved_name}, {country}: "
            f"{temp}°C, wind speed {windspeed} km/h.")

In [6]:
!pip install -q langchain langchain-groq requests
!pip install -U langchain langchain-core langchain-community -q

In [7]:
llm_with_tools = llm.bind_tools([get_weather])

In [8]:
from typing import Annotated
from typing_extensions import TypedDict
from langgraph.graph.message import add_messages

In [9]:
class State(TypedDict):
    messages: Annotated[list, add_messages]

def assistant_node(state: State):
    # Get the conversation history
    current_messages = state["messages"]

    # Ask the LLM what to do
    response = llm_with_tools.invoke(current_messages)
    # Return the response so it gets added to the 'messages' list
    return {"messages": [response]}

In [10]:
tools = [get_weather]

In [11]:
from langgraph.prebuilt import ToolNode

tool_node = ToolNode(tools)

In [12]:
from typing import Literal
from langgraph.graph import END
def should_continue(state: State) -> Literal["tools", END]:
    messages = state["messages"]
    last_message = messages[-1]

    if last_message.tool_calls:
        return "tools"

    else:
        return END

In [13]:
from langgraph.graph import StateGraph, START

# Create a blank graph
builder = StateGraph(State)

In [14]:
from langgraph.graph import StateGraph, START

builder = StateGraph(State)

builder.add_node("assistant", assistant_node)
builder.add_node("tools", tool_node)          # ToolNode handles ALL tools

builder.add_edge(START, "assistant")
builder.add_conditional_edges("assistant", should_continue)
builder.add_edge("tools", "assistant")

In [15]:
graph = builder.compile()

In [16]:
# 1. Define the user's complex request
user_query = "Current weather of varanasi?"

# 2. Run the Graph
initial_state = {"messages": [("user", user_query)]}

print(f"User: {user_query}\n")

# 3. Print the final result
final_response = graph.invoke(initial_state)
print("\n🤖 Final Answer:")
print(final_response["messages"][-1].content)

User: Current weather of varanasi?


🤖 Final Answer:
The current weather in Varanasi is 38.5°C with a wind speed of 12.4 km/h.


## Question 8: Create a LamaIndex implementation that indexes a local text file and retrieves answers from it.


In [17]:
!pip uninstall -y groq
!pip install -q groq==0.31.1

Found existing installation: groq 0.37.1
Uninstalling groq-0.37.1:
  Successfully uninstalled groq-0.37.1


In [18]:
!pip install -q \
llama-index \
llama-index-llms-groq \
langchain-huggingface \
sentence-transformers

In [19]:
text_content = """
LangChain is a framework for developing applications powered by large language models.
It provides tools for chaining together prompts, LLMs, and external data sources.

LlamaIndex is a data framework for connecting custom data sources to large language models.
It specializes in indexing documents and retrieving relevant information for question answering.

Both frameworks are commonly used to build RAG (Retrieval Augmented Generation) applications.
"""

with open("sample.txt", "w") as f:
    f.write(text_content)

In [20]:
from llama_index.core import SimpleDirectoryReader

documents = SimpleDirectoryReader(input_files=["sample.txt"]).load_data()

In [23]:
from llama_index.llms.groq import Groq
from llama_index.core import Settings

llm = Groq(
    model="llama-3.3-70b-versatile",
    api_key=my_api_key
)

Settings.llm = llm

In [25]:
!pip install -q llama-index-embeddings-huggingface

In [26]:
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core import Settings

embed_model = HuggingFaceEmbedding(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

Settings.embed_model = embed_model

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [27]:
from llama_index.core import VectorStoreIndex

index = VectorStoreIndex.from_documents(documents)

In [28]:
query_engine = index.as_query_engine()

response = query_engine.query(
    "What is Llama Index?"
)

print(response)

LlamaIndex is a data framework for connecting custom data sources to large language models, specializing in indexing documents and retrieving relevant information for question answering.


## Question 9: Demonstrate combining Langchain with LamaIndex to create a simple document-based Q&A chatbot.


In [29]:
!pip install -q \
llama-index \
llama-index-llms-groq \
llama-index-embeddings-huggingface \
langchain

In [30]:
from llama_index.core import Settings
from llama_index.llms.groq import Groq
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

Settings.llm = Groq(
    model="llama-3.3-70b-versatile",
    api_key=my_api_key
)

Settings.embed_model = HuggingFaceEmbedding(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

In [31]:
from llama_index.core import SimpleDirectoryReader, VectorStoreIndex

documents = SimpleDirectoryReader(
    input_files=["sample.txt"]
).load_data()

index = VectorStoreIndex.from_documents(documents)
query_engine = index.as_query_engine()

In [33]:
# Creating prompt using LangChain

from langchain_core.prompts import PromptTemplate

prompt = PromptTemplate(
    input_variables=["context", "question"],
    template="""
Answer the question using the provided context.

Context:
{context}

Question:
{question}

Answer:
"""
)

In [35]:
# Building Q & A Chatbot

def chatbot(question):

    # Retrieve relevant information from document
    retrieved = query_engine.query(question)

    context = str(retrieved)

    final_prompt = prompt.format(
        context=context,
        question=question
    )

    response = Settings.llm.complete(final_prompt)

    return response.text

In [36]:
# Asking Question

while True:
    q = input("You: ")

    if q.lower() == "exit":
        break

    answer = chatbot(q)

    print("Bot:", answer)

You: How LlamaIndex is different from LangChain?
Bot: LlamaIndex is different from LangChain in that it specializes in indexing documents and retrieving relevant information for question answering, whereas LangChain is a more general framework for developing applications powered by large language models, focusing on chaining together prompts, LLMs, and external data sources.
You: exit
